# Token-map fusion: layer tokens + Qwen's own per-token view of the question and answer

Step 1 replays every saved prompt + generated answer through the same 4-bit Qwen2.5-7B (teacher forcing, no generation). For each question token, the last prompt token and every answer token it saves hidden states at layers 0, 4, …, 28 (per-layer standardization + 256-dim PCA fitted on source-train rows only) and Qwen's uncertainty for that token: log-probability, entropy / max probability / top-2 margin of the distribution that produced it, argmax hit, next-token entropy and P(<|im_end|>). Two shards run in parallel, one per T4.

Step 2 trains a fusion transformer from scratch over [CLS] + 15 log-sum-exp layer tokens + an answer-uncertainty summary token + one token per text token (PCA states + uncertainty + segment/position embeddings). Text-side tokens are dropped for 20% of training samples.

Runs: **fused** and **layers only** with seeds 42/43/44, plus **text only**. Same six-source split, BCE, 40 epochs, batch 128, best-source-validation-AUROC checkpoint, threshold calibration and held-out TriviaQA evaluation. The fused model is also scored with each side masked, and every answer-level uncertainty feature is reported as a zero-training baseline.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 4d1a1bb
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
TOKEN_DIR = Path('/kaggle/working/token_features')
extract = [sys.executable, '/kaggle/working/SALA/sala_token_features.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(TOKEN_DIR)]
subprocess.run(extract + ['--stage', 'pca', '--device', 'cuda:0'], check=True)
shards = [subprocess.Popen(extract + ['--stage', 'extract', '--shard', str(k), '--num-shards', '2', '--device', f'cuda:{k}']) for k in range(2)]
codes = [p.wait() for p in shards]
if any(codes): raise RuntimeError(f'Token feature extraction failed: {codes}')
print(sorted(p.name for p in TOKEN_DIR.iterdir()))

In [ ]:
RUNS = [('fused', s) for s in (42, 43, 44)] + [('layers', s) for s in (42, 43, 44)] + [('text', 42)]
OUTPUT_DIRS = {}
for mode, seed in RUNS:
    output_dir = Path(f'/kaggle/working/token_map_{mode}_s{seed}')
    OUTPUT_DIRS[(mode, seed)] = output_dir
    command = [sys.executable, '/kaggle/working/SALA/sala_vlm_token_map.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--token-feature-dir', str(TOKEN_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', str(seed), '--train-device', 'cuda:0', '--batch-size', '128', '--mode', mode]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for (mode, seed), output_dir in OUTPUT_DIRS.items():
    s = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'mode': mode, 'seed': seed, 'best_epoch': s['best_epoch'],
                 'source_val_auroc': s['best_source_validation_auroc_during_training'],
                 'source_test_auroc': s['source_test_metrics']['auroc'],
                 'target_auroc': s['target_metrics']['auroc'],
                 'target_balanced_accuracy': s['target_metrics']['balanced_accuracy'],
                 'text_masked_target_auroc': s.get('text_masked_target_auroc'),
                 'layers_masked_target_auroc': s.get('layers_masked_target_auroc')})
table = pd.DataFrame(rows)
display(table)
display(table.groupby('mode')[['source_test_auroc', 'target_auroc', 'target_balanced_accuracy']].agg(['mean', 'std']))
print('Zero-training uncertainty baselines (AUROC, higher value = predicted correct):')
print(json.dumps(s['raw_uncertainty_feature_auroc'], indent=2))